# TABULA Archetype ↔ EPC Matching (Residential)

Map Swedish residential buildings to their **TABULA archetype** based on:
- **Building type**: SFH (småhus / villa / radhus) or MFH (flerbostadshus)
- **Construction year**: mapped to one of 5 TABULA periods

The matched archetype provides: envelope U-values, component areas, construction descriptions, and expected heating demand across 3 Swedish climate zones.

In [1]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path(r"c:\Users\saraabo\Desktop\Project Planning Guide\data\sensitivity\FW_ Map selection in notebook")

with open(DATA_DIR / "tabula_swedish_data.json", encoding="utf-8") as f:
    tabula_envelope = json.load(f)

with open(DATA_DIR / "tabula_webtool_scraped.json", encoding="utf-8") as f:
    tabula_energy = json.load(f)

print(f"Envelope data: {len(tabula_envelope)} archetypes")
print(f"Energy data:   {len(tabula_energy['buildings'])} archetypes")

Envelope data: 10 archetypes
Energy data:   10 archetypes


## 1. Overview of All Archetypes

Build a combined table with envelope properties and energy demand for every archetype.

In [2]:
# ── TABULA construction periods and their year ranges ──
TABULA_PERIODS = [
    ("...1960",    None, 1960),
    ("1961-1975",  1961, 1975),
    ("1976-1985",  1976, 1985),
    ("1986-1995",  1986, 1995),
    ("1996-2005",  1996, 2005),
]

BUILDING_TYPE_LABELS = {"SFH": "Single-Family House (Småhus)", "MFH": "Multi-Family House (Flerbostadshus)"}

# Build combined rows
rows = []
for code, env in tabula_envelope.items():
    energy = tabula_energy["buildings"].get(code, {})
    zones = energy.get("zones", {})
    rows.append({
        "Code": code,
        "Type": env["building_type"],
        "Type Label": BUILDING_TYPE_LABELS.get(env["building_type"], env["building_type"]),
        "Period": env["period"],
        # U-values
        "U Roof": env["u_values"]["roof"],
        "U Wall": env["u_values"]["wall"],
        "U Floor": env["u_values"]["floor"],
        "U Window": env["u_values"]["window"],
        "U Door": env["u_values"]["door"],
        "U Basement Wall": env["u_values"]["basement_wall"],
        # Areas (m²)
        "A Roof": env["areas"]["roof"],
        "A Wall": env["areas"]["wall"],
        "A Floor": env["areas"]["floor"],
        "A Window": env["areas"]["window"],
        "A Door": env["areas"]["door"],
        # Construction
        "Wall Construction": env["construction_types"].get("wall", ""),
        "Roof Construction": env["construction_types"].get("roof", ""),
        "Floor Construction": env["construction_types"].get("floor", ""),
        # Energy demand per zone (kWh/m²/yr)
        "Zone 1 Net (kWh/m²)": zones.get("1", {}).get("net_energy_demand"),
        "Zone 2 Net (kWh/m²)": zones.get("2", {}).get("net_energy_demand"),
        "Zone 3 Net (kWh/m²)": zones.get("3", {}).get("net_energy_demand"),
        "Zone 1 Gross (kWh/m²)": zones.get("1", {}).get("gross_energy_demand_calc"),
        "Zone 2 Gross (kWh/m²)": zones.get("2", {}).get("gross_energy_demand_calc"),
        "Zone 3 Gross (kWh/m²)": zones.get("3", {}).get("gross_energy_demand_calc"),
    })

df_all = pd.DataFrame(rows)
df_all

,Code,Type,Type Label,Period,U Roof,U Wall,U Floor,U Window,U Door,U Basement Wall,...,A Door,Wall Construction,Roof Construction,Floor Construction,Zone 1 Net (kWh/m²),Zone 2 Net (kWh/m²),Zone 3 Net (kWh/m²),Zone 1 Gross (kWh/m²),Zone 2 Gross (kWh/m²),Zone 3 Gross (kWh/m²)
0,SE.N.SFH.01.Gen.ReEx.001,SFH,Single-Family House (Småhus),...1960,0.29,0.60,0.280000,2.34,3.0,0.000000,...,2.0,Murblock i lättbetong (gasbetong),horisontellt vindsbjälklag,platta på mark,198.6,176.2,159.8,389.4,284.2,231.6
1,SE.N.SFH.02.Gen.ReEx.001,SFH,Single-Family House (Småhus),1961-1975,0.21,0.31,0.320000,2.30,2.8,0.000000,...,2.0,Yttervägg småhus 1961-1975,horisontellt vindsbjälklag,platta på mark,188.7,165.7,149.9,343.1,251.1,202.6
2,SE.N.SFH.03.Gen.ReEx.001,SFH,Single-Family House (Småhus),1976-1985,0.15,0.21,0.270000,2.01,2.8,0.000000,...,2.0,Yttervägg småhus 1976- 1985,horisontellt vindsbjälklag,platta på mark,165.2,142.1,123.0,250.3,184.5,148.2
3,SE.N.SFH.04.Gen.ReEx.001,SFH,Single-Family House (Småhus),1986-1995,0.12,0.17,0.240000,1.94,2.8,0.000000,...,2.0,Yttervägg småhus 1986-1995,horisontellt vindsbjälklag,platta på mark,160.1,135.6,116.5,232.0,171.6,137.1
4,SE.N.SFH.05.Gen.ReEx.001,SFH,Single-Family House (Småhus),1996-2005,0.12,0.20,0.180000,1.87,1.5,0.000000,...,2.0,Yttervägg småhus1996-2005,horisontellt vindsbjälklag,platta på mark,156.1,131.1,111.9,219.9,161.9,128.6
5,SE.N.MFH.01.Gen.ReEx.001,MFH,Multi-Family House (Flerbostadshus),...1960,0.36,0.58,0.324910,2.22,3.0,0.702447,...,10.0,Yttervägg flerbostadshus …1960,horisontellt vindsbjälklag,platta på mark,128.8,104.5,87.3,167.3,122.9,98.1
6,SE.N.MFH.02.Gen.ReEx.001,MFH,Multi-Family House (Flerbostadshus),1961-1975,0.20,0.41,0.258303,2.22,2.8,0.702447,...,10.0,Yttervägg flerbostadshus 1961-1975,horisontellt vindsbjälklag,platta på mark,115.3,92.4,76.2,140.6,105.0,82.8
7,SE.N.MFH.03.Gen.ReEx.001,MFH,Multi-Family House (Flerbostadshus),1976-1985,0.17,0.33,0.266789,2.04,2.8,0.702447,...,10.0,Yttervägg flerbostadshus 1976- 1985,horisontellt vindsbjälklag,platta på mark,95.4,75.0,60.8,109.7,81.5,64.0
8,SE.N.MFH.04.Gen.ReEx.001,MFH,Multi-Family House (Flerbostadshus),1986-1995,0.15,0.22,0.241187,1.80,2.8,0.702447,...,10.0,Yttervägg flerbostadshus 1986-1995,horisontellt vindsbjälklag,platta på mark,84.4,66.0,52.9,93.8,70.2,55.1
9,SE.N.MFH.05.Gen.ReEx.001,MFH,Multi-Family House (Flerbostadshus),1996-2005,0.13,0.20,0.206379,1.97,1.5,0.702447,...,10.0,Yttervägg flerbostadshus 1996-2005,horisontellt vindsbjälklag,platta på mark,83.3,65.2,52.1,92.6,69.4,53.7


## 2. U-Value Comparison Across Periods

Heatmap showing how envelope U-values improve over time for each building type.

In [3]:
u_cols = ["U Roof", "U Wall", "U Floor", "U Window", "U Door"]

for btype in ["SFH", "MFH"]:
    subset = df_all[df_all["Type"] == btype].set_index("Period")[u_cols]
    fig = px.imshow(
        subset.values,
        x=u_cols,
        y=subset.index.tolist(),
        color_continuous_scale="YlOrRd",
        aspect="auto",
        title=f"U-Values (W/m²K) — {BUILDING_TYPE_LABELS[btype]}",
        labels={"color": "U-value"},
        text_auto=".2f",
    )
    fig.update_layout(height=350, margin=dict(t=50, b=30))
    fig.show()

## 3. Heating Demand by Archetype & Climate Zone

In [4]:
# Reshape energy data for plotting
energy_rows = []
for _, r in df_all.iterrows():
    for z in [1, 2, 3]:
        energy_rows.append({
            "Archetype": f"{r['Type']} {r['Period']}",
            "Type": r["Type"],
            "Period": r["Period"],
            "Climate Zone": f"Zone {z}",
            "Net Heating Demand (kWh/m²)": r[f"Zone {z} Net (kWh/m²)"],
            "Gross Heating Demand (kWh/m²)": r[f"Zone {z} Gross (kWh/m²)"],
        })

df_energy = pd.DataFrame(energy_rows)

fig = px.bar(
    df_energy,
    x="Archetype", y="Net Heating Demand (kWh/m²)",
    color="Climate Zone",
    barmode="group",
    title="Net Heating Demand by Archetype and Climate Zone",
    color_discrete_sequence=["#2563eb", "#8AB62E", "#f59e0b"],
)
fig.update_layout(height=450, xaxis_tickangle=-35, margin=dict(b=100))
fig.show()

## 4. EPC → TABULA Archetype Matching Function

Given a building type and construction year from an EPC, find the corresponding TABULA archetype.

In [5]:
def year_to_tabula_period(year: int) -> str:
    """Map a construction year to the TABULA period label."""
    if year <= 1960:
        return "...1960"
    elif year <= 1975:
        return "1961-1975"
    elif year <= 1985:
        return "1976-1985"
    elif year <= 1995:
        return "1986-1995"
    elif year <= 2005:
        return "1996-2005"
    else:
        return None  # Post-2005: no TABULA archetype available


def epc_to_building_type(epc_category: str) -> str:
    """
    Map an EPC building category string to SFH or MFH.
    Handles common Swedish EPC labels.
    """
    cat = epc_category.strip().lower()
    sfh_keywords = ["småhus", "villa", "radhus", "kedjehus", "parhus",
                    "friliggande", "single-family", "sfh"]
    mfh_keywords = ["flerbostadshus", "lägenhet", "apartment",
                    "multi-family", "mfh", "hyreshus", "bostadsrätt"]
    for kw in sfh_keywords:
        if kw in cat:
            return "SFH"
    for kw in mfh_keywords:
        if kw in cat:
            return "MFH"
    return None  # Unknown type


def match_archetype(building_type: str, construction_year: int) -> dict | None:
    """
    Return the full TABULA archetype (envelope + energy) for a given
    building type (SFH/MFH) and construction year.
    """
    btype = building_type.upper()
    if btype not in ("SFH", "MFH"):
        btype = epc_to_building_type(building_type)
    if not btype:
        return None

    period = year_to_tabula_period(construction_year)
    if not period:
        return None

    # Find the matching archetype code
    match = df_all[(df_all["Type"] == btype) & (df_all["Period"] == period)]
    if match.empty:
        return None

    row = match.iloc[0]
    return {
        "code": row["Code"],
        "building_type": btype,
        "type_label": BUILDING_TYPE_LABELS[btype],
        "period": period,
        "u_values": {c.replace("U ", "").lower(): row[c] for c in u_cols},
        "areas_m2": {
            "roof": row["A Roof"], "wall": row["A Wall"],
            "floor": row["A Floor"], "window": row["A Window"], "door": row["A Door"],
        },
        "construction": {
            "wall": row["Wall Construction"],
            "roof": row["Roof Construction"],
            "floor": row["Floor Construction"],
        },
        "heating_demand_net_kWh_m2": {
            "zone_1": row["Zone 1 Net (kWh/m²)"],
            "zone_2": row["Zone 2 Net (kWh/m²)"],
            "zone_3": row["Zone 3 Net (kWh/m²)"],
        },
        "heating_demand_gross_kWh_m2": {
            "zone_1": row["Zone 1 Gross (kWh/m²)"],
            "zone_2": row["Zone 2 Gross (kWh/m²)"],
            "zone_3": row["Zone 3 Gross (kWh/m²)"],
        },
    }

print("Matching functions ready.")

Matching functions ready.


## 5. Example Lookup

Try a few example buildings to see which TABULA archetype they map to.

In [6]:
examples = [
    ("Småhus (villa)", 1955),
    ("Flerbostadshus", 1972),
    ("Radhus",         1983),
    ("Flerbostadshus", 1990),
    ("Villa",          2002),
    ("Flerbostadshus", 2015),  # Post-2005 → no match
]

for epc_cat, year in examples:
    result = match_archetype(epc_cat, year)
    if result:
        print(f"\n{'='*60}")
        print(f"EPC: {epc_cat}, built {year}")
        print(f"  -> Archetype: {result['code']}")
        print(f"     {result['type_label']}  |  Period: {result['period']}")
        print(f"     U-values: wall={result['u_values']['wall']}, "
              f"roof={result['u_values']['roof']}, "
              f"window={result['u_values']['window']}, "
              f"floor={result['u_values']['floor']}")
        print(f"     Heating demand (net): "
              f"Z1={result['heating_demand_net_kWh_m2']['zone_1']}, "
              f"Z2={result['heating_demand_net_kWh_m2']['zone_2']}, "
              f"Z3={result['heating_demand_net_kWh_m2']['zone_3']} kWh/m2")
    else:
        btype = epc_to_building_type(epc_cat)
        period = year_to_tabula_period(year)
        print(f"\n{'='*60}")
        print(f"EPC: {epc_cat}, built {year}")
        print(f"  -> NO MATCH (type={btype}, period={period})")


EPC: Småhus (villa), built 1955
  -> Archetype: SE.N.SFH.01.Gen.ReEx.001
     Single-Family House (Småhus)  |  Period: ...1960
     U-values: wall=0.6, roof=0.29, window=2.34, floor=0.28
     Heating demand (net): Z1=198.6, Z2=176.2, Z3=159.8 kWh/m2

EPC: Flerbostadshus, built 1972
  -> Archetype: SE.N.MFH.02.Gen.ReEx.001
     Multi-Family House (Flerbostadshus)  |  Period: 1961-1975
     U-values: wall=0.41, roof=0.2, window=2.22, floor=0.25830258302583
     Heating demand (net): Z1=115.3, Z2=92.4, Z3=76.2 kWh/m2

EPC: Radhus, built 1983
  -> Archetype: SE.N.SFH.03.Gen.ReEx.001
     Single-Family House (Småhus)  |  Period: 1976-1985
     U-values: wall=0.21, roof=0.15, window=2.01, floor=0.27
     Heating demand (net): Z1=165.2, Z2=142.1, Z3=123.0 kWh/m2

EPC: Flerbostadshus, built 1990
  -> Archetype: SE.N.MFH.04.Gen.ReEx.001
     Multi-Family House (Flerbostadshus)  |  Period: 1986-1995
     U-values: wall=0.22, roof=0.15, window=1.8, floor=0.241187384044527
     Heating demand (ne

## 6. Batch EPC Matching

If you have a DataFrame of EPC records with columns like `building_category` and `construction_year`, use this to classify all of them at once.

In [7]:
def classify_epc_batch(epc_df: pd.DataFrame,
                       type_col: str = "building_category",
                       year_col: str = "construction_year") -> pd.DataFrame:
    """
    Add TABULA archetype columns to an EPC DataFrame.
    Returns the original DataFrame with new columns:
      tabula_type, tabula_period, tabula_code,
      u_wall, u_roof, u_window, u_floor,
      heating_net_z1, heating_net_z2, heating_net_z3
    """
    results = []
    for _, row in epc_df.iterrows():
        arch = match_archetype(str(row[type_col]), int(row[year_col]))
        if arch:
            results.append({
                "tabula_type": arch["building_type"],
                "tabula_period": arch["period"],
                "tabula_code": arch["code"],
                "u_wall": arch["u_values"]["wall"],
                "u_roof": arch["u_values"]["roof"],
                "u_window": arch["u_values"]["window"],
                "u_floor": arch["u_values"]["floor"],
                "heating_net_z1": arch["heating_demand_net_kWh_m2"]["zone_1"],
                "heating_net_z2": arch["heating_demand_net_kWh_m2"]["zone_2"],
                "heating_net_z3": arch["heating_demand_net_kWh_m2"]["zone_3"],
            })
        else:
            results.append({
                "tabula_type": None, "tabula_period": None, "tabula_code": None,
                "u_wall": None, "u_roof": None, "u_window": None, "u_floor": None,
                "heating_net_z1": None, "heating_net_z2": None, "heating_net_z3": None,
            })

    return pd.concat([epc_df.reset_index(drop=True),
                      pd.DataFrame(results)], axis=1)


# Demo with sample EPC-like data
sample_epcs = pd.DataFrame([
    {"address": "Storgatan 10",  "building_category": "Flerbostadshus", "construction_year": 1968, "epc_energy": 145},
    {"address": "Björkvägen 3",  "building_category": "Småhus (villa)",  "construction_year": 1952, "epc_energy": 180},
    {"address": "Tallvägen 7",   "building_category": "Radhus",          "construction_year": 1991, "epc_energy": 110},
    {"address": "Ekgatan 22",    "building_category": "Flerbostadshus", "construction_year": 1999, "epc_energy":  72},
    {"address": "Nybyggarv. 1",  "building_category": "Villa",           "construction_year": 2012, "epc_energy":  65},
])

result_df = classify_epc_batch(sample_epcs)
result_df

,address,building_category,construction_year,epc_energy,tabula_type,tabula_period,tabula_code,u_wall,u_roof,u_window,u_floor,heating_net_z1,heating_net_z2,heating_net_z3
0,Storgatan 10,Flerbostadshus,1968,145,MFH,1961-1975,SE.N.MFH.02.Gen.ReEx.001,0.41,0.20,2.22,0.258303,115.3,92.4,76.2
1,Björkvägen 3,Småhus (villa),1952,180,SFH,...1960,SE.N.SFH.01.Gen.ReEx.001,0.60,0.29,2.34,0.280000,198.6,176.2,159.8
2,Tallvägen 7,Radhus,1991,110,SFH,1986-1995,SE.N.SFH.04.Gen.ReEx.001,0.17,0.12,1.94,0.240000,160.1,135.6,116.5
3,Ekgatan 22,Flerbostadshus,1999,72,MFH,1996-2005,SE.N.MFH.05.Gen.ReEx.001,0.20,0.13,1.97,0.206379,83.3,65.2,52.1
4,Nybyggarv. 1,Villa,2012,65,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Compare EPC Actual vs TABULA Expected

Where the EPC measured energy differs significantly from the TABULA expected demand, it may indicate renovation has already happened — or that the building is underperforming.

In [8]:
# Use Zone 3 (south Sweden) as comparison baseline — adjust if needed
comparison = result_df.dropna(subset=["tabula_code"]).copy()
comparison["tabula_expected"] = comparison["heating_net_z3"]
comparison["delta"] = comparison["epc_energy"] - comparison["tabula_expected"]
comparison["delta_pct"] = (comparison["delta"] / comparison["tabula_expected"] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=comparison["address"], y=comparison["epc_energy"],
    name="EPC Actual", marker_color="#2563eb",
))
fig.add_trace(go.Bar(
    x=comparison["address"], y=comparison["tabula_expected"],
    name="TABULA Expected (Zone 3)", marker_color="#8AB62E",
))
fig.update_layout(
    title="EPC Actual vs TABULA Expected Heating Demand",
    yaxis_title="kWh/m²/year",
    barmode="group", height=400,
)
fig.show()

print("\nDelta (EPC - TABULA):")
comparison[["address", "building_category", "construction_year",
            "epc_energy", "tabula_expected", "delta", "delta_pct"]]


Delta (EPC - TABULA):


,address,building_category,construction_year,epc_energy,tabula_expected,delta,delta_pct
0,Storgatan 10,Flerbostadshus,1968,145,76.2,68.8,90.3
1,Björkvägen 3,Småhus (villa),1952,180,159.8,20.2,12.6
2,Tallvägen 7,Radhus,1991,110,116.5,-6.5,-5.6
3,Ekgatan 22,Flerbostadshus,1999,72,52.1,19.9,38.2
